# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malakanwarr/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
Our K-Means model identified distinct performance archetypes across the portfolio. To make this actionable for the content team, we isolate the specific cluster representing the "Missed Opportunity" archetype (high search volume, low click capture) and generate a prioritized queue.

* The Action: REVIEW_AND_UPDATE - Directs the content team to expand thin sections, update stale facts, and improve scanability.

* The Reason Code: MISSED_OPPORTUNITY_ARCHETYPE - Explains why the page was flagged, building trust by showing the AI found a massive gap between demand and actual traffic.

* The Ranking: Pages are ranked by their proximity (urgency score) to the exact mathematical center of this cluster, ensuring the most severe cases sit at the top of the queue.

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the full dataset
df = pd.read_csv('master_dataset_ready.csv', low_memory=False)

# 2. Re-run our K-Means model to get our archetypes
feature_cols = ['search_volume', 'gsc_clicks', 'competition', 'keyword_char_count']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols].fillna(0))

kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
df['cluster'] = kmeans.fit_predict(X_scaled)

# 3. Identify the "Missed Opportunity" cluster (the one with the highest concentration of high volume/low clicks)
df['temp_target'] = ((df['search_volume'] >= 1000) & (df['gsc_clicks'] <= 5)).astype(int)
target_cluster = df.groupby('cluster')['temp_target'].mean().idxmax()

# 4. Score every page by how close it is to that specific archetype's center
distances = kmeans.transform(X_scaled)[:, target_cluster]
df['urgency_score'] = -distances  # Negative distance so a higher score = closer to the center

# 5. Assign Human-Readable Actions and Reason Codes
df['action'] = 'IGNORE'
df['reason_code'] = 'HEALTHY_OR_LOW_PRIORITY'

is_target = df['cluster'] == target_cluster
df.loc[is_target, 'action'] = 'REVIEW_AND_UPDATE'
df.loc[is_target, 'reason_code'] = 'MISSED_OPPORTUNITY_ARCHETYPE'

# 6. Build the Ranked Queue (Filter to only the ones needing review, sort highest urgency to the top)
ranked_queue = df[df['action'] == 'REVIEW_AND_UPDATE'].sort_values(by='urgency_score', ascending=False)

print(f"Queue successfully generated! Found {len(ranked_queue)} pages assigned to the Missed Opportunity archetype.")
print("Here are the Top 5 most urgent pages for the content team to fix Monday morning:")
display(ranked_queue[['content_hash_id', 'search_volume', 'gsc_clicks', 'action', 'reason_code', 'urgency_score']].head(5))

Queue successfully generated! Found 2 pages assigned to the Missed Opportunity archetype.
Here are the Top 5 most urgent pages for the content team to fix Monday morning:


,content_hash_id,search_volume,gsc_clicks,action,reason_code,urgency_score
29032,content_b9ffa30eb293951f,368000.0,0.0,REVIEW_AND_UPDATE,MISSED_OPPORTUNITY_ARCHETYPE,-2.144218
33714,content_04e4047dc8eef2fd,368000.0,0.0,REVIEW_AND_UPDATE,MISSED_OPPORTUNITY_ARCHETYPE,-2.144218


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: This playbook is a decision-support tool designed to provide directional guidance for the content optimization queue. By highlighting observed patterns in historical performance, it helps the content team prioritize pages that mathematically exhibit a gap between search demand and click capture.

Limits: This model relies entirely on measured historical data from a static snapshot (March 2026). It does not predict future Google algorithm updates, shifting user intent, or seasonal demand spikes. Furthermore, the model is blind to specific business value; it cannot distinguish between a low-volume informational blog post and a low-volume high-ticket enterprise sales page.

In [2]:
# Calculate the limits of the queue's scope
total_portfolio_pages = len(df)
queue_size = len(ranked_queue)
filtered_percentage = (queue_size / total_portfolio_pages) * 100

print("--- Tool Scope & Limits ---")
print(f"Total pages in portfolio: {total_portfolio_pages:,}")
print(f"Pages routed to human review: {queue_size:,}")
print(f"Noise filtered: The model successfully filters out {100 - filtered_percentage:.4f}% of the portfolio.")
print("Limit Check: The model only flags mathematical gaps. A human must still verify if the search volume aligns with actual business intent.")


--- Tool Scope & Limits ---
Total pages in portfolio: 46,649
Pages routed to human review: 2
Noise filtered: The model successfully filters out 99.9957% of the portfolio.
Limit Check: The model only flags mathematical gaps. A human must still verify if the search volume aligns with actual business intent.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
Mandatory Human Review: Before acting on the REVIEW_AND_UPDATE queue, a human editor must verify Intent Match (does the high search volume actually align with the business purpose of the page?) and Business Value (is this an SEO-driven asset, or a dedicated landing page for paid ads where organic clicks don't matter?).

The No-Go List (Never Automate):
1. Pruning/Deletions: Never automate the deletion or redirecting of a page based on low AI scores.
2. Navigational Hubs: Never alter login screens, shopping carts, or account portals to "optimize" for SEO.
3. Legal/Compliance: Privacy policies and terms of service must be exempt from all automated content recommendations.

In [3]:
# 1. Check for 'navigational' intent pages that should be exempt from standard SEO actions
if 'main_intent' in df.columns:
    no_go_pages = df[df['main_intent'] == 'navigational']

    print("--- The NO-GO List ---")
    print(f"Found {len(no_go_pages)} 'navigational' pages in the portfolio (e.g., login screens, portals).")
    print("WARNING: These pages must NEVER be automatically flagged for SEO content expansion or pruning.")

    if len(no_go_pages) > 0:
        display(no_go_pages[['content_hash_id', 'search_volume', 'gsc_clicks', 'main_intent']].head(5))
else:
    print("--- The NO-GO List ---")
    print("No 'main_intent' column found to filter navigational pages.")
    print("Alternative Rule: A URL substring filter (e.g., '/login', '/cart', '/privacy') must be applied before routing to writers.")

--- The NO-GO List ---
Found 43 'navigational' pages in the portfolio (e.g., login screens, portals).


,content_hash_id,search_volume,gsc_clicks,main_intent
455,content_01b4fc122fc4f394,210.0,0.0,navigational
609,content_0309dcc478ad93f6,90.0,0.0,navigational
714,content_04077371c99cb578,30.0,0.0,navigational
1044,content_06de5345d1c3b9a8,390.0,0.0,navigational
1107,content_0778f816747e9925,50.0,0.0,navigational


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The archetypes identified by this K-Means model are based entirely on search behavior from the March 2026 snapshot. These recommendations will go stale if macroeconomic search behavior shifts (for example, if AI search engines like ChatGPT or Perplexity begin capturing a massive share of informational queries, permanently lowering traditional search volume).

The Retrain Trigger: We will monitor the portfolio's median search volume and average click capture. If the median search volume drops by more than 20% from our current baseline, it indicates a structural shift in search behavior. At that point, the current cluster boundaries become invalid, and the model must be retrained on fresh data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The prioritized content queue has been generated and exported to the local work/outputs/ directory. By design, this CSV file remains out of version control to protect client data, but the logic to regenerate it on demand is preserved in this notebook. This export serves as the foundational artifact for the final research paper and the operational handoff to the content team.

In [4]:
import os

# 1. Ensure the outputs directory exists
os.makedirs('work/outputs', exist_ok=True)

# 2. Export the ranked queue we built in Section 1
export_path = 'work/outputs/action_playbook_queue.csv'
ranked_queue.to_csv(export_path, index=False)

print("--- Export Complete ---")
print(f"Ranked queue successfully saved to: {export_path}")
print("This file is ready for the content team and the final research paper.")

--- Export Complete ---
Ranked queue successfully saved to: work/outputs/action_playbook_queue.csv
This file is ready for the content team and the final research paper.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.